In [1]:
import pickle

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

In [14]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("new_taxi_experiment_deployment")

2025/05/03 17:56:33 INFO mlflow.tracking.fluent: Experiment with name 'new_taxi_experiment_deployment' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-models-deployment/2', creation_time=1746309393878, experiment_id='2', last_update_time=1746309393878, lifecycle_stage='active', name='new_taxi_experiment_deployment', tags={}>

In [15]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [16]:
df_train = read_dataframe('s3://mlops-zoomcamp-hal4zhou/data_source/green_tripdata_2023-01.parquet')
df_val = read_dataframe('s3://mlops-zoomcamp-hal4zhou/data_source/green_tripdata_2023-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [17]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = root_mean_squared_error(y_pred, y_val)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 5.39920274232368


2025/05/03 17:57:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run adventurous-seal-464 at: http://127.0.0.1:5000/#/experiments/2/runs/baf244d99b5049c28758ca0de4b42a02
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [18]:
mlflow.pyfunc.get_model_dependencies("runs:/baf244d99b5049c28758ca0de4b42a02/model")

2025/05/03 18:07:44 INFO mlflow.pyfunc: To install the dependencies that were used to train the model, run the following command: '%pip install -r /var/folders/xv/p1z69rt949l1ttxwv1f6vr340000gn/T/tmpobqa0wfe/model/requirements.txt'.


'/var/folders/xv/p1z69rt949l1ttxwv1f6vr340000gn/T/tmpobqa0wfe/model/requirements.txt'

In [10]:
from mlflow.tracking import MlflowClient


In [20]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = 'b4d3bca8aa8e46a6b8257fe4541b1136'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [21]:
path = client.download_artifacts(run_id=RUN_ID, path='dict_vectorizer.bin')

In [22]:
with open(path, 'rb') as f_out:
    dv = pickle.load(f_out)

In [23]:
dv

DictVectorizer()